# PCA Trajectory Analysis for Kolmogorov Flow

This notebook analyzes PCA reconstruction quality on individual trajectories for Re 40 and Re 5000.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import os
from sklearn.decomposition import PCA

plt.style.use('seaborn-v0_8-darkgrid')
%matplotlib inline


In [ ]:
def load_kolmogorov_trajectories(scratch_dir, reynolds=40, num_trajectories=10):
    """Load Kolmogorov flow trajectories from data file."""
    possible_paths = [
        os.path.join(scratch_dir, 'data', f'2D_NS_Re{reynolds}.npy'),
        os.path.join(scratch_dir, f'2D_NS_Re{reynolds}.npy'),
        f'/pscratch/sd/d/dvod/data/2D_NS_Re{reynolds}.npy',
        f'/pscratch/sd/d/dvod/2D_NS_Re{reynolds}.npy',
    ]
    
    data_path = None
    for path in possible_paths:
        if os.path.exists(path):
            data_path = path
            break
    
    if data_path is None:
        raise FileNotFoundError(f"Dataset not found. Tried: {possible_paths}")
    
    print(f"Loading Kolmogorov flow Re {reynolds} from: {data_path}")
    data = np.load(data_path, mmap_mode='r')
    
    if len(data.shape) == 4:
        num_traj_total, num_timesteps, H, W = data.shape
        num_trajectories = min(num_trajectories, num_traj_total)
        
        trajectories = []
        for traj_idx in range(num_trajectories):
            traj_data = np.array(data[traj_idx])
            trajectories.append(traj_data)
        
        return np.array(trajectories), num_timesteps, H, W
    else:
        raise ValueError(f"Unexpected data shape: {data.shape}")


In [ ]:
def compute_pca_on_trajectories(trajectories, n_components=30):
    """Compute PCA on all trajectories."""
    num_traj, num_timesteps, H, W = trajectories.shape
    
    data_flat = trajectories.reshape(-1, H * W).astype(np.float32)
    
    pca = PCA(n_components=n_components)
    pca.fit(data_flat)
    
    return pca, data_flat.shape


In [ ]:
def reconstruct_trajectory(pca, trajectory, n_components=None):
    """Reconstruct a single trajectory using PCA."""
    num_timesteps, H, W = trajectory.shape
    trajectory_flat = trajectory.reshape(num_timesteps, H * W).astype(np.float32)
    
    if n_components is None:
        n_components = pca.n_components_
    
    coeffs = pca.transform(trajectory_flat)[:, :n_components]
    
    reconstructed_flat = pca.inverse_transform(
        np.pad(coeffs, ((0, 0), (0, pca.n_components_ - n_components)), mode='constant')
    )
    reconstructed = reconstructed_flat.reshape(num_timesteps, H, W)
    
    return reconstructed, coeffs

def compute_fourier_modes(trajectories, n_fourier_modes=30):
    """Compute Fourier basis modes from trajectories."""
    num_traj, num_timesteps, H, W = trajectories.shape
    
    # Compute 2D FFT for all timesteps
    # FFT returns modes ordered by frequency: [0, 1, 2, ..., N/2, -N/2+1, ..., -1]
    # We select the top n_fourier_modes by magnitude
    
    # Compute FFT for all data
    fft_data = np.fft.fft2(trajectories, axes=(-2, -1))
    fft_magnitudes = np.abs(fft_data)
    
    # Flatten to (num_traj * num_timesteps, H * W) for mode selection
    fft_flat = fft_data.reshape(-1, H, W)
    mag_flat = fft_magnitudes.reshape(-1, H, W)
    
    # Find most important modes (by average magnitude across all snapshots)
    mode_importance = np.mean(mag_flat, axis=0)
    mode_indices_flat = np.argsort(mode_importance.ravel())[::-1][:n_fourier_modes]
    mode_indices_2d = np.unravel_index(mode_indices_flat, (H, W))
    
    # Store mode indices
    fourier_mode_indices = list(zip(mode_indices_2d[0], mode_indices_2d[1]))
    
    return fourier_mode_indices, mode_importance

def reconstruct_trajectory_fourier(trajectory, fourier_mode_indices, n_fourier_modes=None):
    """Reconstruct a single trajectory using Fourier basis."""
    num_timesteps, H, W = trajectory.shape
    
    if n_fourier_modes is None:
        n_fourier_modes = len(fourier_mode_indices)
    
    # Compute FFT
    fft_traj = np.fft.fft2(trajectory, axes=(-2, -1))
    
    # Create mask for selected modes
    mode_mask = np.zeros((H, W), dtype=bool)
    for i, (k, l) in enumerate(fourier_mode_indices[:n_fourier_modes]):
        mode_mask[k, l] = True
    
    # Keep only selected modes
    fft_filtered = np.zeros_like(fft_traj)
    for t in range(num_timesteps):
        fft_filtered[t] = np.where(mode_mask, fft_traj[t], 0)
    
    # Inverse FFT to reconstruct
    reconstructed = np.real(np.fft.ifft2(fft_filtered, axes=(-2, -1)))
    
    # Extract coefficients (magnitude and phase of selected modes)
    coeffs = []
    for k, l in fourier_mode_indices[:n_fourier_modes]:
        mode_coeffs = fft_traj[:, k, l]
        coeffs.append(mode_coeffs)
    coeffs = np.array(coeffs).T  # (num_timesteps, n_fourier_modes)
    
    return reconstructed, coeffs


In [ ]:
def compute_metrics(original, reconstructed):
    """Compute MAE, RMSE, Rel L1, Rel L2 for each timestep."""
    num_timesteps = original.shape[0]
    
    mae = np.zeros(num_timesteps)
    rmse = np.zeros(num_timesteps)
    rel_l1 = np.zeros(num_timesteps)
    rel_l2 = np.zeros(num_timesteps)
    
    for t in range(num_timesteps):
        orig_t = original[t].flatten()
        recon_t = reconstructed[t].flatten()
        
        error = orig_t - recon_t
        
        mae[t] = np.mean(np.abs(error))
        rmse[t] = np.sqrt(np.mean(error ** 2))
        
        l1_norm_orig = np.sum(np.abs(orig_t))
        l2_norm_orig = np.sqrt(np.sum(orig_t ** 2))
        
        if l1_norm_orig > 1e-10:
            rel_l1[t] = np.sum(np.abs(error)) / l1_norm_orig
        else:
            rel_l1[t] = 0.0
        
        if l2_norm_orig > 1e-10:
            rel_l2[t] = np.sqrt(np.sum(error ** 2)) / l2_norm_orig
        else:
            rel_l2[t] = 0.0
    
    return mae, rmse, rel_l1, rel_l2


In [ ]:
def plot_trajectory_comparison(original, reconstructed, deviation, title_prefix=""):
    """Plot original, reconstructed, and deviation in 3 columns."""
    num_timesteps, H, W = original.shape
    
    n_plot = min(10, num_timesteps)
    timestep_indices = np.linspace(0, num_timesteps - 1, n_plot, dtype=int)
    
    fig, axes = plt.subplots(n_plot, 3, figsize=(15, 3 * n_plot))
    if n_plot == 1:
        axes = axes.reshape(1, -1)
    
    vmin_orig = original.min()
    vmax_orig = original.max()
    vmin_dev = deviation.min()
    vmax_dev = deviation.max()
    
    for i, t_idx in enumerate(timestep_indices):
        im1 = axes[i, 0].imshow(original[t_idx], cmap='viridis', aspect='auto', vmin=vmin_orig, vmax=vmax_orig)
        axes[i, 0].set_title(f'Original t={t_idx}')
        axes[i, 0].axis('off')
        plt.colorbar(im1, ax=axes[i, 0])
        
        im2 = axes[i, 1].imshow(reconstructed[t_idx], cmap='viridis', aspect='auto', vmin=vmin_orig, vmax=vmax_orig)
        axes[i, 1].set_title(f'PCA Reconstructed t={t_idx}')
        axes[i, 1].axis('off')
        plt.colorbar(im2, ax=axes[i, 1])
        
        im3 = axes[i, 2].imshow(deviation[t_idx], cmap='RdBu_r', aspect='auto', vmin=vmin_dev, vmax=vmax_dev)
        axes[i, 2].set_title(f'Deviation t={t_idx}')
        axes[i, 2].axis('off')
        plt.colorbar(im3, ax=axes[i, 2])
    
    plt.suptitle(f'{title_prefix}Trajectory Comparison: Original vs PCA Reconstruction', fontsize=14, y=0.995)
    plt.tight_layout()
    return fig


In [ ]:
def plot_metrics_vs_time(mae, rmse, rel_l1, rel_l2, title_prefix=""):
    """Plot metrics vs time."""
    fig, axes = plt.subplots(2, 2, figsize=(12, 10))
    
    time_steps = np.arange(len(mae))
    
    axes[0, 0].plot(time_steps, mae, 'b-', linewidth=2)
    axes[0, 0].set_xlabel('Time Step')
    axes[0, 0].set_ylabel('MAE')
    axes[0, 0].set_title('Mean Absolute Error vs Time')
    axes[0, 0].grid(True, alpha=0.3)
    
    axes[0, 1].plot(time_steps, rmse, 'r-', linewidth=2)
    axes[0, 1].set_xlabel('Time Step')
    axes[0, 1].set_ylabel('RMSE')
    axes[0, 1].set_title('Root Mean Squared Error vs Time')
    axes[0, 1].grid(True, alpha=0.3)
    
    axes[1, 0].plot(time_steps, rel_l1 * 100, 'g-', linewidth=2)
    axes[1, 0].set_xlabel('Time Step')
    axes[1, 0].set_ylabel('Relative L1 (%)')
    axes[1, 0].set_title('Relative L1 Error vs Time')
    axes[1, 0].grid(True, alpha=0.3)
    
    axes[1, 1].plot(time_steps, rel_l2 * 100, 'm-', linewidth=2)
    axes[1, 1].set_xlabel('Time Step')
    axes[1, 1].set_ylabel('Relative L2 (%)')
    axes[1, 1].set_title('Relative L2 Error vs Time')
    axes[1, 1].grid(True, alpha=0.3)
    
    plt.suptitle(f'{title_prefix}PCA Reconstruction Error Metrics vs Time', fontsize=14)
    plt.tight_layout()
    return fig


In [ ]:
scratch_dir = '/pscratch/sd/dvod'
num_trajectories = 10
n_pca_modes = 120
n_fourier_modes = 30
reynolds_list = [40, 5000]


In [ ]:
for reynolds in reynolds_list:
    print(f"\n{'='*80}")
    print(f"Processing Re {reynolds}")
    print(f"{'='*80}")
    
    trajectories, num_timesteps, H, W = load_kolmogorov_trajectories(
        scratch_dir, reynolds=reynolds, num_trajectories=num_trajectories
    )
    print(f"Loaded {len(trajectories)} trajectories, each with {num_timesteps} timesteps, size {H}x{W}")
    
    # PCA Analysis
    print(f"\n{'='*60}")
    print(f"PCA Analysis with {n_pca_modes} modes")
    print(f"{'='*60}")
    print(f"Computing PCA with {n_pca_modes} modes...")
    pca, data_shape = compute_pca_on_trajectories(trajectories, n_components=n_pca_modes)
    print(f"PCA explained variance ratio (first {n_pca_modes} modes): {pca.explained_variance_ratio_.sum():.4f}")
    
    traj_idx = np.random.randint(0, len(trajectories))
    print(f"\nSelected trajectory index: {traj_idx}")
    
    selected_trajectory = trajectories[traj_idx]
    
    reconstructed_pca, coeffs_pca = reconstruct_trajectory(pca, selected_trajectory, n_components=n_pca_modes)
    
    deviation_pca = selected_trajectory - reconstructed_pca
    
    mae_pca, rmse_pca, rel_l1_pca, rel_l2_pca = compute_metrics(selected_trajectory, reconstructed_pca)
    
    print(f"\nPCA Average metrics over trajectory:")
    print(f"  MAE: {mae_pca.mean():.6f}")
    print(f"  RMSE: {rmse_pca.mean():.6f}")
    print(f"  Rel L1: {rel_l1_pca.mean()*100:.2f}%")
    print(f"  Rel L2: {rel_l2_pca.mean()*100:.2f}%")
    
    title_prefix = f"Re {reynolds} - PCA ({n_pca_modes} modes) - "
    fig1 = plot_trajectory_comparison(
        selected_trajectory, reconstructed_pca, deviation_pca, title_prefix=title_prefix
    )
    plt.savefig(f'pca_trajectory_comparison_Re{reynolds}.png', dpi=150, bbox_inches='tight')
    plt.show()
    
    fig2 = plot_metrics_vs_time(mae_pca, rmse_pca, rel_l1_pca, rel_l2_pca, title_prefix=title_prefix)
    plt.savefig(f'pca_metrics_vs_time_Re{reynolds}.png', dpi=150, bbox_inches='tight')
    plt.show()
    
    # Fourier Analysis
    print(f"\n{'='*60}")
    print(f"Fourier Analysis with {n_fourier_modes} modes")
    print(f"{'='*60}")
    print(f"Computing Fourier modes with {n_fourier_modes} modes...")
    fourier_mode_indices, mode_importance = compute_fourier_modes(trajectories, n_fourier_modes=n_fourier_modes)
    print(f"Selected {len(fourier_mode_indices)} Fourier modes")
    
    reconstructed_fourier, coeffs_fourier = reconstruct_trajectory_fourier(
        selected_trajectory, fourier_mode_indices, n_fourier_modes=n_fourier_modes
    )
    
    deviation_fourier = selected_trajectory - reconstructed_fourier
    
    mae_fourier, rmse_fourier, rel_l1_fourier, rel_l2_fourier = compute_metrics(
        selected_trajectory, reconstructed_fourier
    )
    
    print(f"\nFourier Average metrics over trajectory:")
    print(f"  MAE: {mae_fourier.mean():.6f}")
    print(f"  RMSE: {rmse_fourier.mean():.6f}")
    print(f"  Rel L1: {rel_l1_fourier.mean()*100:.2f}%")
    print(f"  Rel L2: {rel_l2_fourier.mean()*100:.2f}%")
    
    title_prefix = f"Re {reynolds} - Fourier ({n_fourier_modes} modes) - "
    fig3 = plot_trajectory_comparison(
        selected_trajectory, reconstructed_fourier, deviation_fourier, title_prefix=title_prefix
    )
    plt.savefig(f'fourier_trajectory_comparison_Re{reynolds}.png', dpi=150, bbox_inches='tight')
    plt.show()
    
    fig4 = plot_metrics_vs_time(
        mae_fourier, rmse_fourier, rel_l1_fourier, rel_l2_fourier, title_prefix=title_prefix
    )
    plt.savefig(f'fourier_metrics_vs_time_Re{reynolds}.png', dpi=150, bbox_inches='tight')
    plt.show()
    
    print(f"\nPlots saved for Re {reynolds}")
    print(f"  PCA: pca_trajectory_comparison_Re{reynolds}.png, pca_metrics_vs_time_Re{reynolds}.png")
    print(f"  Fourier: fourier_trajectory_comparison_Re{reynolds}.png, fourier_metrics_vs_time_Re{reynolds}.png")
